In [1]:
import requests
import json
import re
from IPython.display import JSON


from music21 import key, chord, stream, pitch, note, tempo, meter, interval, clef
from fractions import Fraction

In [2]:
# url = "https://api.hooktheory.com/v1/songs/public/Rzoj_pYQmDe"
url = "https://api.hooktheory.com/v1/songs/public/Abm_awLPxak"


In [3]:
response = requests.get(url)

In [50]:
# metadata = response.json()

In [4]:
json_data = response.json()['jsonData']

In [5]:
json_object = json.loads(json_data)

In [6]:
json_object['chords']

[{'root': 1,
  'beat': 1,
  'duration': 4,
  'type': 5,
  'inversion': 0,
  'applied': 0,
  'adds': [],
  'omits': [],
  'alterations': [],
  'suspensions': [],
  'pedal': None,
  'alternate': '',
  'borrowed': None,
  'isRest': False,
  'recordingEndBeat': None},
 {'root': 5,
  'beat': 5,
  'duration': 3.5,
  'type': 5,
  'inversion': 0,
  'applied': 0,
  'adds': [],
  'omits': [],
  'alterations': [],
  'suspensions': [],
  'pedal': None,
  'alternate': '',
  'borrowed': None,
  'isRest': False,
  'recordingEndBeat': None},
 {'root': 1,
  'beat': 8.5,
  'duration': 0.5,
  'type': 5,
  'inversion': 2,
  'applied': 0,
  'adds': [],
  'omits': [],
  'alterations': [],
  'suspensions': [],
  'pedal': None,
  'alternate': '',
  'borrowed': '',
  'isRest': False,
  'recordingEndBeat': None},
 {'root': 5,
  'beat': 9,
  'duration': 1,
  'type': 7,
  'inversion': 0,
  'applied': 0,
  'adds': [],
  'omits': [],
  'alterations': [],
  'suspensions': [],
  'pedal': None,
  'alternate': '',
  'b

In [8]:
JSON(json_object, expanded=True, indent=2)

<IPython.core.display.JSON object>

In [6]:
RHYTHM_PRESETS = {
    "default": [(0.0, 4.0)],
    "pop": [
        (0.0, 0.5), (0.5, 0.5), (1.0, 0.5), (1.5, 0.5),
        (2.0, 0.5), (3.0, 1.0)
    ],
    "ballad": [(0.0, 2.0), (2.0, 2.0)],
    "offbeat": [(0.5, 0.5), (2.5, 0.5)],
    "syncopated": [(0.0, 0.5), (1.25, 0.5), (2.5, 0.5), (3.75, 0.25)]
}

def apply_chord_rhythm(ch, beat_offset, rhythm_pattern):
    notes = []
    for rel_offset, dur in rhythm_pattern:
        # Clone the chord for each rhythm hit
        ch_copy = chord.Chord(ch.pitches)
        ch_copy.duration.quarterLength = dur
        ch_copy.offset = beat_offset + rel_offset
        notes.append(ch_copy)
    return notes


# 1. Scale degree to pitch
def degree_to_pitch(degree, key_obj):
    sc = key_obj.getScale()
    return sc.pitchFromDegree(degree)

# 2. Build basic chord
def build_basic_chord(root_pitch, chord_type):
    if chord_type == 5:  # Major triad
        return chord.Chord([root_pitch, root_pitch.transpose('M3'), root_pitch.transpose('P5')])
    elif chord_type == 2:  # Minor triad
        return chord.Chord([root_pitch, root_pitch.transpose('m3'), root_pitch.transpose('P5')])
    elif chord_type == 7:  # Dominant 7
        return chord.Chord([root_pitch, root_pitch.transpose('M3'), root_pitch.transpose('P5'), root_pitch.transpose('m7')])
    elif chord_type == 4:  # Minor 7
        return chord.Chord([root_pitch, root_pitch.transpose('m3'), root_pitch.transpose('P5'), root_pitch.transpose('m7')])
    elif chord_type == 1:  # Diminished triad
        return chord.Chord([root_pitch, root_pitch.transpose('m3'), root_pitch.transpose('d5')])
    else:
        return chord.Chord([root_pitch, root_pitch.transpose('M3'), root_pitch.transpose('P5')])  # Default major

# 3. Apply modifiers
def apply_modifications(ch, adds, omits, alterations, suspensions):
    # Suspensions (replace 3rd)
    if suspensions:
        ch = chord.Chord([p for p in ch.pitches if interval.notesToInterval(ch.root(), p).semitones not in [3, 4]])
        for sus in suspensions:
            if sus == 2:
                ch.add(ch.root().transpose('M2'))
            elif sus == 4:
                ch.add(ch.root().transpose('P4'))

    # Omits
    for o in omits:
        ch = chord.Chord([p for p in ch.pitches if interval.notesToInterval(ch.root(), p).semitones != o])

    # Adds
    for a in adds:
        ch.add(ch.root().transpose(a))

    # Alterations (as semitone offsets)
    for alt in alterations:
        ch.add(ch.root().transpose(alt))

    return ch

# 4. Set inversion
def set_inversion(ch, inversion):
    if inversion > 0 and inversion < len(ch.pitches):
        bass = ch.pitches[inversion]
        ch.bass(note.Note(bass))
    return ch

# 5. Full parser
def parse_json_chord(json_chord, key_obj):
    root_pitch = degree_to_pitch(json_chord['root'], key_obj)
    ch = build_basic_chord(root_pitch, json_chord['type'])

    ch = apply_modifications(
        ch,
        adds=json_chord.get('adds', []),
        omits=json_chord.get('omits', []),
        alterations=json_chord.get('alterations', []),
        suspensions=json_chord.get('suspensions', [])
    )

    ch = set_inversion(ch, json_chord.get('inversion', 0))
    ch.quarterLength = json_chord.get('duration', 1)
    return ch

# 6. Apply swing
def apply_swing(beat_offset, swing_factor=0.0, swing_unit=0.5):
    """
    Adjusts the timing of a beat offset based on swing.
    Applies swing to events that start on the 'off' part of the beat unit.
    E.g., with swing_unit=0.5, events on x.5 will be delayed.
    """
    # base = int(beat_offset / swing_unit)
    remainder = beat_offset % swing_unit

    # Only apply swing to the second note of the beat unit (e.g., off-beat eighths)
    if abs(remainder - swing_unit) < 1e-6:
        swing_amount = swing_factor * (swing_unit / 2)
        return beat_offset + swing_amount
    return beat_offset

# def get_current_event(beat, event_list):
#     """
#     Find the latest event that occurs at or before the given beat.
#     """
#     result = event_list[0]
#     for i, e in enumerate(event_list):
#         if beat >= e['beat']:
#             result = event_list[i]
#             break

#     return result

# 7. Build the score
def parse_chords(chord_list, key_info, tempo_info, rhythm="default"):
    s = stream.Part()
    s.id = "Chords"
    # Key
    tonic = key_info['tonic']
    scale_mode = key_info['scale']
    key_obj = key.Key(tonic=tonic, mode=scale_mode)
    # s.append(key_obj)

    # Tempo
    # bpm = tempo_info['bpm']
    # mm = tempo.MetronomeMark(number=bpm)
    # s.append(mm)

    # Meter
    # ts = meter.TimeSignature(f"{meter_info['numBeats']}/{int(1 / meter_info['beatUnit'])}")
    # s.append(ts)

    swing_factor = tempo_info.get("swingFactor", 0)
    swing_beat = tempo_info.get("swingBeat", 0.5)

    # Chords
    for jc in chord_list:
        offset = apply_swing(jc['beat'], swing_factor, swing_beat)

        if not jc.get('isRest', False):
            ch = parse_json_chord(jc, key_obj)
        else:
            ch = note.Rest(quarterLength=jc.get('duration', 1))

        # Rhythm Preset
        pattern = RHYTHM_PRESETS.get(rhythm, [(0.0, ch.quarterLength)])

        for ch_event in apply_chord_rhythm(ch, offset, pattern):
            s.insert(ch_event.offset, ch_event)
        # ch.offset = beat_offset
        # s.insert(ch.offset, ch)

    return s

def sd_to_pitch(sd, key_obj, octave_offset):
    """
    Converts solfège degree (e.g., 'b7', '#4', '3') to a pitch.Pitch object,
    based on a music21.key.Key object and octave offset.
    """
    # Match optional accidentals ('b', '#', 'bb', '##') followed by a digit (1–7)
    match = re.fullmatch(r'([b#]*)([1-7])', sd)
    if not match:
        raise ValueError(f"Invalid scale degree format: {sd}")
    
    accidental_str, degree_str = match.groups()
    degree = int(degree_str)

    # Diatonic semitone offsets for major scale
    scale_degrees = {
        1: 0, 2: 2, 3: 4, 4: 5, 5: 7, 6: 9, 7: 11
    }

    semitones = scale_degrees[degree]

    # Apply accidentals
    for accidental in accidental_str:
        if accidental == '#':
            semitones += 1
        elif accidental == 'b':
            semitones -= 1

    # Get tonic pitch
    tonic_pitch = pitch.Pitch(key_obj.tonic.name)
    midi_number = tonic_pitch.midi + semitones + 12 * octave_offset
    return pitch.Pitch(midi=midi_number)

def parse_melody(melody_data, key_info):
    s = stream.Part()
    s.id = "Melody"
    # Key
    tonic = key_info['tonic']
    scale_mode = key_info['scale']
    key_obj = key.Key(tonic=tonic, mode=scale_mode)

    for n in melody_data:
        duration = n['duration']
        offset = n['beat']
    
        if n.get('isRest', False):
            m_note = note.Rest()
        else:
            octave = n.get('octave', 0)
            sd = n['sd']
            p = sd_to_pitch(sd, key_obj, octave)
            m_note = note.Note(p)

        m_note.quarterLength = duration
        s.insert(offset, m_note)

    return s

def write_combined_score(chord_data, melody_data, key_info, tempo_info, meter_info, rhythm="default"):
    score = stream.Score()

    # Key
    tonic = key_info['tonic']
    scale_mode = key_info['scale']
    key_obj = key.Key(tonic=tonic, mode=scale_mode)
    score.append(key_obj)

    # Tempo
    bpm = tempo_info['bpm']
    mm = tempo.MetronomeMark(number=bpm)
    score.append(mm)

    # Meter
    ts = meter.TimeSignature(f"{meter_info['numBeats']}/{int(4 * meter_info['beatUnit'])}")
    score.append(ts)

    chords_part = parse_chords(chord_data, key_info, tempo_info, rhythm)
    melody_part = parse_melody(melody_data, key_info)

    melody_part.insert(0, clef.TrebleClef())
    chords_part.insert(0, clef.BassClef())

    score.insert(0, melody_part)
    score.insert(0, chords_part)

    return score


In [9]:


# Example JSON data
chords = json_object['chords']
melody = json_object['notes']
keys = json_object['keys'][0]
tempos = json_object['tempos'][0]
meters = json_object['meters'][0]
rhythm = 'ballad'  # or 'pop', 'ballad', etc.

score = write_combined_score(chords, melody, keys, tempos, meters, rhythm)
score.write('midi', fp='results/output.mid')
print("MIDI file saved as chord_progression_output.mid")

MIDI file saved as chord_progression_output.mid
